# Example: Bag of Words, TF-IDF, and PMI
In this example, we'll play around with simple collections of text data and explore how to create basic text embeddings using the Bag of Words model, Term Frequency-Inverse Document Frequency (TF-IDF), and Pointwise Mutual Information (PMI).

> __Learning Objectives:__
> 
> By the end of this example, you should be able to:
>
> * __Bag of Words construction__: Construct Bag of Words count vectors for text data using a vocabulary dictionary that maps words to indices. Compare this dictionary-based approach to a hashing-based vectorizer that avoids maintaining an explicit vocabulary.
> * __TF-IDF computation__: Compute term frequency and smoothed inverse document frequency values for each word in a corpus. Combine them into TF-IDF scores and use cosine similarity to compare document representations.
> * __PMI estimation__: Build a word co-occurrence matrix from a corpus using a fixed context window. Compute PMI and Positive PMI (PPMI) values from the co-occurrence statistics to quantify word associations.

Let's get started!
___

## Setup, Data, and Prerequisites
We set up the computational environment by including the `Include.jl` file, loading any needed resources, such as sample datasets, and setting up any required constants.

> The `Include.jl` file also loads external packages, various functions that we will use in the exercise, and custom types to model the components of our problem. It checks for a `Manifest.toml` file; if it finds one, packages are loaded. Other packages are downloaded and then loaded.

In [1]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

### Data
Here, let's construct a small corpus of simple sentences to work with in the `sentences::Array{String,1}` variable.

In [2]:
sentences = let

    # initialize -
    sentences_array = Array{String,1}(); # initialize an array of sentence strings

    # add sentences -
    push!(sentences_array, "I love machine learning and data science ."); #1 the first sentence is different than the others
    push!(sentences_array, "Machine learning is fun ."); # the second sentence is close to the third sentence
    push!(sentences_array, "Machine learning is great ."); # expect similarity b/w 2nd and 3rd sentences
    push!(sentences_array, "I love coding machine learning in Julia !"); # 4 the fourth sentence is similar to the first sentence
    push!(sentences_array, "Julia is a great programming language ?");
    push!(sentences_array, "I enjoy learning new things about data science , machine learning , and artificial intelligence .");
    
    # return the array
    sentences_array
end

6-element Vector{String}:
 "I love machine learning and data science ."
 "Machine learning is fun ."
 "Machine learning is great ."
 "I love coding machine learning in Julia !"
 "Julia is a great programming language ?"
 "I enjoy learning new things abo" ⋯ 35 bytes ⋯ ", and artificial intelligence ."

Next, we'll preprocess the data in the `sentences::Array{String,1}` variable by tokenizing the sentences into words, and converting them to lowercase. This, along with our control tokens, will be our vocabulary model.

We'll store the vocabulary in the `vocabulary::Dict{String, Int64}` variable, where the keys are the unique words and the values are their corresponding indices, and the inverse vocabulary in the `inverse_vocabulary::Dict{Int64, String}` variable, where the keys are the indices and the values are the unique words.

In [3]:
vocabulary, inverse_vocabulary = let

    # initialize -
    vocabulary = Dict{String, Int64}(); # initialize the vocabulary dictionary
    inverse_vocabulary = Dict{Int64, String}(); # initialize the inverse vocabulary dictionary
    index = 1; # initialize the index counter

    # control tokens -
    control_tokens = ["<bos>", "<eos>", "<pad>", "<unk>"]; # define control tokens

    # tmp variables -
    words = Set{String}(); # temporary array to hold words
    for sentence in sentences
        tmp = split(lowercase(sentence)); # convert to lowercase and split by whitespace
        push!(words, tmp...); # add words to the set
    end
    words_array = collect(words) |> sort; # convert set to array, and sort it

    # append the control tokens to the words array
    words_array = vcat(words_array, control_tokens);
    for word in words_array
        vocabulary[word] = index;
        inverse_vocabulary[index] = word;
        index += 1;
    end

    # return the vocabulary and inverse vocabulary
    (vocabulary, inverse_vocabulary)
end

(Dict("!" => 1, "is" => 17, "enjoy" => 11, "data" => 10, "language" => 19, "coding" => 9, "science" => 25, "<bos>" => 27, "a" => 5, "and" => 7…), Dict(5 => "a", 16 => "intelligence", 20 => "learning", 12 => "fun", 24 => "programming", 28 => "<eos>", 8 => "artificial", 17 => "is", 30 => "<unk>", 1 => "!"…))

What's in the `vocabulary::Dict{String, Int64}` and `inverse_vocabulary::Dict{Int64, String}` variables? Let's take a look!

In [4]:
vocabulary

Dict{String, Int64} with 30 entries:
  "!"            => 1
  "is"           => 17
  "enjoy"        => 11
  "data"         => 10
  "language"     => 19
  "coding"       => 9
  "science"      => 25
  "<bos>"        => 27
  "a"            => 5
  "and"          => 7
  ","            => 2
  "programming"  => 24
  "love"         => 21
  "?"            => 4
  "."            => 3
  "in"           => 15
  "i"            => 14
  "<unk>"        => 30
  "about"        => 6
  "<pad>"        => 29
  "machine"      => 22
  "artificial"   => 8
  "learning"     => 20
  "intelligence" => 16
  "great"        => 13
  ⋮              => ⋮

What can we do with these vocabulary models? One thing we can do is take a sentence, and convert it into an index vector using the vocabulary model. Let's check that out!

In [5]:
let

    # initialize -
    i = 1; # select a sentence index to inspect
    sentence = sentences[i]; # get the sentence

    augmented_sentence = "<bos> " * sentence * " <eos>";
    words = split(lowercase(augmented_sentence)) .|> String;
    word_indices = [get(vocabulary, word, vocabulary["<unk>"]) for word in words]; # wow! nice one-liner to get indices with <unk> fallback

    @show sentence;
    @show words;
    @show word_indices;
end;

sentence = "I love machine learning and data science ."
words = ["<bos>", "i", "love", "machine", "learning", "and", "data", "science", ".", "<eos>"]
word_indices = [27, 14, 21, 22, 20, 7, 10, 25, 3, 28]


### Helper Implementations
The [`hashing_vectorizer(...)` function](src/BagOfWords.jl) converts a list of string features (words) into a fixed-length count vector using the hashing trick. Each feature is mapped to a vector index via `mod1(abs(hash(f)), length)`, and the count at that index is incremented. Multiple features that hash to the same index are summed (a collision).

___

## Task 1: Bag of Words Representations
In this task, we'll create a Bag of Words representation for our corpus of sentences. 

> __Bag of Words Model__
>
> The Bag of Words (BoW) model is a technique for text embedding. As the name suggests, we represent a text (such as a sentence or a document) as a "bag" (multiset) of its words, disregarding grammar and word order but keeping multiplicity. Given a vocabulary of unique words, each text is represented as a vector where each dimension corresponds to a word in the vocabulary, and the value in that dimension indicates the frequency of that word in the text.

The [`build_bow_matrix(...)` function](src/BagOfWords.jl) iterates over each sentence, lowercases and tokenizes it, wraps it with `<bos>` and `<eos>` boundary tokens, and counts each token into a row of the output matrix. It returns an integer matrix of size `num_sentences × vocab_size`.

Let's compute the Bag of Words representation for each sentence in our corpus and store the results in the `bow_matrix::Array{Int64, 2}` variable, where each row corresponds to a sentence and each column corresponds to a word in the vocabulary.

In [ ]:
bow_matrix = build_bow_matrix(sentences, vocabulary)

### What is wrong with this representation? 
There are some issues with the Bag of Words representation.

> __So what's the problem?__
>
> We had text, now we have numerical vectors. Great! Why are you complaining?
> 
> * __Sparsity and dimension:__ The counts require us to maintain a vocabulary dictionary that maps words to indices in our vectors. But suppose we have a very large vocabulary. Further, most of the entries in our vectors are zero. This is called a "sparse" representation, and it can be inefficient in terms of both storage and computation. 
> 
> * __Order:__ We also have to impose some order on our vocabulary (e.g., alphabetical order) to ensure that the indices are consistent across different texts. This can be cumbersome, especially when dealing with large and dynamic vocabularies.
> 
> Let's consider an alternative representation. 

Instead of maintaining a vocabulary dictionary, the [`hashing_vectorizer(...)` function](src/BagOfWords.jl) builds a vector of a pre-defined length by applying [Julia's built-in `hash(...)` function](https://docs.julialang.org/en/v1/base/base/#Base.hash) to each word and using the result directly as a feature index. This avoids maintaining an explicit vocabulary, at the cost of potential hash collisions.

> __Hash function?__
> 
> A hash function is a function that takes an input (or 'message') and returns a fixed-size string of bytes. The output, typically a hash code or hash value, is deterministic (same input always produces the same output) but not necessarily unique (different inputs can produce the same hash, called a collision). Hash functions are commonly used in computer science for tasks such as data retrieval, cryptography, and data integrity verification.

We save this output in the `vectorized_sentence::Array{Int64,1}` variable. Let's take a look!

In [ ]:
vectorized_sentence = let
    i = 1;
    sentence = sentences[i];
    augmented_sentence = "<bos> " * sentence * " <eos>";
    words = split(lowercase(augmented_sentence)) .|> String;
    hashing_vectorizer(words, length = length(vocabulary))
end

You can change the sentence index `i` in the code block above to see how different sentences are represented using this hashing vectorizer. Further, you can modify the `length` parameter to see how the size of the output vector affects the representation.

Play around with different sentences and vector lengths to see how the hashing vectorizer performs! We'll save this output in the `alternative_vectorized_sentence::Array{Int64,1}` variable. Let's take a look!

In [ ]:
alternative_vectorized_sentence = let
    i = 2;
    sentence = sentences[i];
    augmented_sentence = "<bos> " * sentence * " <eos>";
    words = split(lowercase(augmented_sentence)) .|> String;
    hashing_vectorizer(words, length = 10)
end

### Hash Collisions
When using the hashing trick, multiple words may map to the same index in the output vector. This is called a **collision**.

> __What happens with collisions?__
>
> *   **Information loss**: When two different words map to the same index, their counts are combined, and the model cannot distinguish between them.
> *   **Dimensionality trade-off**: Smaller vector lengths increase the probability of collisions but reduce memory usage. Larger vector lengths reduce collisions but increase sparsity and memory usage.
> *   **Mitigation**: In practice, we choose a vector length large enough (e.g., $2^{18}$ or $2^{20}$) to minimize collisions for the given vocabulary size.

___

## Task 2: TF-IDF Representations
In this task, we'll compute the Term Frequency-Inverse Document Frequency (TF-IDF) representation for our corpus of sentences. The TF-IDF score for a term $t$ in a document $d$ is given by the product of two terms:
$$
\boxed{
\begin{align*}
\text{TF-IDF}(t, d) &= \text{tf}(t, d) \cdot \text{idf}(t, \mathcal{D})
\end{align*}}
$$

The __Term Frequency__ ($\text{tf}$) is the raw count of term $t$ in document $d$, often normalized by the total number of words in $d$. The __Inverse Document Frequency__ ($\text{idf}$) measures how much the term is tied to a subset of documents. It is calculated as:
    $$ \text{idf}(t, \mathcal{D}) = \ln \left( \frac{N}{|\{d \in \mathcal{D} : t \in d\}|} \right) $$
where $N$ is the total number of documents in the corpus $\mathcal{D}$, and the denominator is the number of documents where the term $t$ appears. In practice (especially for small corpora), we often use a smoothed IDF to avoid division-by-zero and to reduce extreme values; we'll use that smoothed form below.

The [`build_tf_matrix(...)` function](src/TFIDF.jl) divides each entry in the BoW count matrix by the total token count for that sentence, returning a matrix of relative term frequencies of size `num_sentences × vocab_size`.

Let's compute the TF values for each term in our vocabulary for each sentence in the corpus and store them in the `TF_matrix::Array{Float64, 2}` variable, where each row corresponds to a sentence and each column corresponds to a word in the vocabulary.

In [ ]:
TF_matrix = build_tf_matrix(bow_matrix)

The [`build_idf_dictionary(...)` function](src/TFIDF.jl) computes a smoothed IDF value for each vocabulary token by counting the number of sentences in which that token appears ($df$) and returning $\ln\!\left(\frac{N+1}{df+1}\right)$, where $N$ is the total number of sentences. It returns a `Dict{String, Float64}` mapping each token to its IDF score.

To keep IDF values finite in small corpora (and to avoid division-by-zero when a term appears in zero documents), we'll use a smoothed IDF:
$$
\boxed{
\begin{align*}
\text{idf}(t, \mathcal{D}) &= \ln \left( \frac{N + 1}{|\{d \in \mathcal{D} : t \in d\}| + 1} \right)
\end{align*}}
$$
where $N$ is the total number of documents in the corpus $\mathcal{D}$. Let's compute the IDF values for each term in our vocabulary across the entire corpus and store them in the `IDF_values_dictionary::Dict{String, Float64}` variable.

In [ ]:
IDF_values_dictionary = build_idf_dictionary(bow_matrix, vocabulary, length(sentences))

The [`build_tfidf_matrix(...)` function](src/TFIDF.jl) multiplies each column of the TF matrix by the corresponding IDF value from `IDF_values_dictionary`, returning a TF-IDF matrix of size `num_sentences × vocab_size`. Tokens absent from the IDF dictionary are assigned a score of zero.

Let's compute the TF-IDF representation for each sentence and store it in the `TFIDF_matrix::Array{Float64, 2}` variable, where each row corresponds to a sentence and each column corresponds to a word in the vocabulary.

In [ ]:
TFIDF_matrix = build_tfidf_matrix(TF_matrix, IDF_values_dictionary, inverse_vocabulary)

So what does the TF-IDF representation tell us about our sentences? Let's compute the similarity between the TF-IDF vectors of two sentences to see how similar they are in terms of their content. 

> __Test:__ Let's compute the cosine similarity between the TF and the TF-IDF vectors of sentence 2 and sentence 3 (we know these sentences share some words) versus sentence 1 and sentence 5 (we know these sentences do not share any words). Vector 2 and 3 should be more similar than Vector 1 and 5.

What do we get?

In [13]:
let

    # initialize -
    i = 2; # index of first sentence to inspect
    j = 3; # index of second sentence to inspect
    D = TFIDF_matrix; # you can use TF-IDF or TF matrix for this example

    # get the TF-IDF vectors for the two sentences
    vᵢ = D[i, :];
    vⱼ = D[j, :];

    dot_product = dot(vᵢ, vⱼ);
    magnitude_vᵢ = norm(vᵢ);
    magnitude_vⱼ = norm(vⱼ);
    
    if magnitude_vᵢ == 0 || magnitude_vⱼ == 0
        return 0.0 # Return 0 for vectors with zero magnitude
    end
    
    value = dot_product / (magnitude_vᵢ * magnitude_vⱼ);
    println("Cosine similarity between sentence $i and sentence $j: $value");
end

Cosine similarity between sentence 2 and sentence 3: 0.3036826972774764


___

## Task 3: PMI Representations
In this task, we'll compute a Pointwise Mutual Information (PMI) representation for our corpus using word co-occurrence counts within a fixed context window.

> __Pointwise Mutual Information (PMI)__
>
> PMI compares how often a word $w$ and a context word $c$ appear together relative to how often we would expect them to co-occur if they were independent.

The PMI between a word $w$ and a context word $c$ is defined as:
$$
\boxed{
\begin{align*}
\text{PMI}(w, c) &= \log_2 \frac{P(w, c)}{P(w)P(c)}
\end{align*}}
$$
where $P(w, c)$ is the probability of observing $w$ and $c$ in the same window, and $P(w)$ and $P(c)$ are the corresponding marginal probabilities. We'll estimate these probabilities from word-context co-occurrence counts in the corpus.

The [`build_cooccurrence_matrix(...)` function](src/PMI.jl) iterates over each sentence, lowercases and tokenizes it, and for every word increments the count at `[w_idx, ctx_idx]` for each context word within `window_size` positions. It returns a symmetric integer matrix of size `vocab_size × vocab_size`.

Let's build a word-context co-occurrence matrix using a window size of $m=2$ tokens on each side and store the counts in the `cooccurrence_matrix::Array{Int64, 2}` variable, where rows correspond to target words and columns correspond to context words.

In [ ]:
cooccurrence_matrix = build_cooccurrence_matrix(sentences, vocabulary, window_size = 2)

The [`build_pmi_matrices(...)` function](src/PMI.jl) estimates joint and marginal probabilities from the co-occurrence counts, then computes $\log_2\!\left(P(w,c)\,/\,(P(w)\,P(c))\right)$ for each word-context pair. It returns two matrices of size `vocab_size × vocab_size`: the full PMI matrix (with $-\infty$ for zero co-occurrence pairs) and the PPMI matrix (negative values clamped to zero).

Let's convert the co-occurrence counts to probabilities and compute PMI. We'll store the PMI values in the `PMI_matrix::Array{Float64, 2}` variable, and the Positive PMI values in the `PPMI_matrix::Array{Float64, 2}` variable.

In [ ]:
PMI_matrix, PPMI_matrix = build_pmi_matrices(cooccurrence_matrix);

Now we can inspect the PMI or PPMI matrices to see which word pairs co-occur more than expected in this small corpus.

In [16]:
PPMI_matrix

30×30 Matrix{Float64}:
 0.0      0.0      0.0      0.0      0.0      …  0.0       2.53051  0.0  0.0
 0.0      0.0      0.0      0.0      0.0         0.0       0.0      0.0  0.0
 0.0      0.0      0.0      0.0      0.0         0.0       2.53051  0.0  0.0
 0.0      0.0      0.0      0.0      0.0         0.0       2.53051  0.0  0.0
 0.0      0.0      0.0      0.0      0.0         0.0       0.0      0.0  0.0
 0.0      0.0      0.0      0.0      0.0      …  0.0       0.0      0.0  0.0
 0.0      1.70044  0.0      0.0      0.0         0.0       0.0      0.0  0.0
 0.0      2.70044  2.11548  0.0      0.0         0.0       0.0      0.0  0.0
 0.0      0.0      0.0      0.0      0.0         0.0       0.0      0.0  0.0
 0.0      1.70044  1.11548  0.0      0.0         0.0       0.0      0.0  0.0
 0.0      0.0      0.0      0.0      0.0      …  2.11548   0.0      0.0  0.0
 0.0      0.0      2.11548  0.0      0.0         0.0       2.11548  0.0  0.0
 0.0      0.0      1.11548  0.0      2.70044     0.0 

### Understanding Window Size and Co-occurrence
Two words "co-occur" in PMI if they appear within a specified window size of each other, not just in the same sentence.

> __Important:__ With `window_size = 2`, a word only "sees" the 2 tokens immediately before and 2 tokens immediately after it. Words that are more than 2 positions apart do NOT co-occur, even if they're in the same sentence.
>
> For example, in "I love coding machine learning in Julia", the words "love" and "julia" are 5 positions apart, so they don't co-occur with `window_size = 2`.

Let's test this by comparing two cases:

In [19]:
let
    # Example 1: Words that are FAR apart in a sentence (window_size = 2)
    word_1 = "love" |> lowercase;
    word_2 = "julia" |> lowercase;
    index_1 = vocabulary[word_1];
    index_2 = vocabulary[word_2];
    
    # Check the co-occurrence count
    cooccurrence_count = cooccurrence_matrix[index_1, index_2];
    ppmi_value = PPMI_matrix[index_1, index_2];
    
    println("Example 1: \"$word_1\" and \"$word_2\":");
    println("  Co-occurrence count: $cooccurrence_count");
    println("  PPMI value: $ppmi_value");
    println("  Note: These words appear in sentence 4 but are 5 positions apart (beyond window_size=2)\n");
    
    # Example 2: Words that ARE within the window
    word_3 = "machine" |> lowercase;
    word_4 = "learning" |> lowercase;
    index_3 = vocabulary[word_3];
    index_4 = vocabulary[word_4];
    
    cooccurrence_count_2 = cooccurrence_matrix[index_3, index_4];
    ppmi_value_2 = PPMI_matrix[index_3, index_4];
    
    println("Example 2: \"$word_3\" and \"$word_4\":");
    println("  Co-occurrence count: $cooccurrence_count_2");
    println("  PPMI value: $ppmi_value_2");
    println("  Note: These words appear together frequently within window_size=2");
end

Example 1: "love" and "julia":
  Co-occurrence count: 0
  PPMI value: 0.0
  Note: These words appear in sentence 4 but are 5 positions apart (beyond window_size=2)

Example 2: "machine" and "learning":
  Co-occurrence count: 5
  PPMI value: 1.267480310864986
  Note: These words appear together frequently within window_size=2


___

## Summary
In this example, we implemented and explored three text embedding techniques: Bag of Words, TF-IDF, and PMI.

> __Key Takeaways__:
> 
> * __Bag of Words limitations__: BoW provides a frequency-based representation, but most entries are zero for large vocabularies, resulting in sparse high-dimensional vectors. The model also discards word order and does not capture semantic similarity between words.
> * __TF-IDF re-weighting__: TF-IDF reduces the influence of common words by scaling term counts with inverse document frequency. This produces vectors that are more useful for similarity comparisons between documents.
> * __PMI for associations__: PMI and PPMI quantify whether two words co-occur more or less than expected by chance within a fixed window. The window size determines which word pairs count as co-occurring, so distant words in the same sentence may have zero co-occurrence.

These count-based methods form the foundation for more advanced text embedding techniques.
___